In [ ]:
import torch
from d2l import torch as d2l
from torch import nn

In [ ]:
#@save
## 绘制heatmap，反应QK的相似度
def show_heatmaps(matrices, xlabel, ylabel, titles=None, figsize=(2.5, 2.5),
                  cmap='Reds'):
    """显示矩阵热图"""
    d2l.use_svg_display()
    num_rows, num_cols = matrices.shape[0], matrices.shape[1]
    fig, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize,
                                 sharex=True, sharey=True, squeeze=False)
    for i, (row_axes, row_matrices) in enumerate(zip(axes, matrices)):
        for j, (ax, matrix) in enumerate(zip(row_axes, row_matrices)):
            pcm = ax.imshow(matrix.detach().numpy(), cmap=cmap)
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    fig.colorbar(pcm, ax=axes, shrink=0.6);

In [ ]:
attention_weights = torch.eye(10).reshape((1, 1, 10, 10))
show_heatmaps(attention_weights, xlabel='K', ylabel='Q')

In [ ]:
## dataset
n_train = 50
x_train,_ = torch.sort(torch.rand(n_train) * 5)
def f(x):
    return 2 * torch.sin(x) + x ** 0.8

y_train = f(x_train) + torch.normal(0.0,0.5,(n_train,))
x_test = torch.arange(0,5,0.1)

y_truth = f(x_test)
n_test = len(x_test)



In [ ]:
def plot_kernel_reg(y_hat):
    ## 绘制真实曲线和预测曲线
    d2l.plot(x_test, [y_truth, y_hat], 'x', 'y', 
             legend=['Truth', 'Pred'],
             xlim=[0, 5], ylim=[-1, 5])
    
    ## 叠加训练数据的散点图
    d2l.plt.plot(x_train, y_train, 'o', alpha=0.5)

In [ ]:
## average pooling as baseline model
y_hat = torch.repeat_interleave(y_train.mean(),n_test)
plot_kernel_reg(y_hat)

In [ ]:
## nadaraya_watson kernel regression using Gaussian kernel
x_repeat = x_test.repeat_interleave(n_train).reshape((-1,n_train))

attention_weights = nn.functional.softmax(
    -(x_repeat - x_train)**2 / 2,
    dim=1
)
y_hat = torch.matmul(attention_weights,y_train)

plot_kernel_reg(y_hat)

In [ ]:
d2l.show_heatmaps(attention_weights.unsqueeze(0).unsqueeze(0),xlabel='Sorted training inputs',ylabel='Sorted testing inputs')

In [ ]:
## Batch matrix multiplication
weights = torch.ones((2,10))*0.1
values = torch.arange(20.0).reshape((2,10))
torch.bmm(
    weights.unsqueeze(1),
    values.unsqueeze(-1)
)

In [ ]:
## parametric NW kernel regressiom
class NWKernelRegression(nn.Module):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.w = nn.Parameter(torch.rand((1,),requires_grad=True))
    def forward(self,queries,keys,values):
        ## 将 Q shape 调整为（batch_size,n_train）
        queries = queries.repeat_interleave(keys.shape[1]).reshape((-1,keys.shape[1]))
        self.attention_weights = nn.functional.softmax(
            -((queries-keys)*self.w)**2 / 2,
            dim=1
        )
        return torch.bmm(
            self.attention_weights.unsqueeze(1),
            values.unsqueeze(-1)
        ).reshape(-1) ## (n,1)
    

In [ ]:
## 留一法
X_tile = x_train.repeat((n_train,1))
Y_tile = y_train.repeat((n_train,1))

keys = X_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train,-1))

values = Y_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train,-1))



In [ ]:
net = NWKernelRegression()
loss = nn.MSELoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=0.5)
animator = d2l.Animator(xlabel='epoch', ylabel='loss', xlim=[1, 5])
for epoch in range(5):
    trainer.zero_grad()
    l = loss(net(x_train, keys, values), y_train).sum()
    l += 1e-3 * torch.norm(net.w)  # L2 norm
    l.sum().backward()
    trainer.step()
    print(f'epoch {epoch + 1}, loss {float(l.sum()):.6f}')
    animator.add(epoch + 1, float(l.sum()))

In [ ]:
## keys的形状:(n_test，n_train)，
keys = x_train.repeat((n_test, 1))
## values的形状:(n_test，n_train)
values = y_train.repeat((n_test, 1))
y_hat = net(x_test, keys, values).unsqueeze(1).detach()
plot_kernel_reg(y_hat)

In [ ]:
d2l.show_heatmaps(net.attention_weights.unsqueeze(0).unsqueeze(0),xlabel='Sorted training inputs',ylabel='Sorted testing inputs')